# Complex feature: linguistic markers as features
SW7 project, group S7G5, AAU Copenhagen.

At the meeting on 10 Nov our supervisor asked us to test one more complex linguistic feature. The rule-based check blocks a prompt as soon as one of its nine rules fires. Here the nine rules become on/off features, and we add eight text-shape features from GuardChain (Majhi et al. 2026). A learned model gives each marker a weight. We add the markers to Char TF-IDF + SVM, our tier 1 pick, and also try them on their own.

Same test prompts and training data as `S7G5_Classifiers.ipynb`. Everything runs on the CPU in a few minutes. The code is in `complex_feature.py`.

## 1. Setup

In [ ]:
try:   # reload the .py files after you edit them (fails on Google Colab's older IPython, which is fine)
    %load_ext autoreload
    %autoreload 2
except ModuleNotFoundError:
    pass

from huggingface_hub import get_token

from benchmark import Benchmark
from classifiers import rule_check, train_sklearn
from complex_feature import changed, marker_models, marker_weights, summary
from prompt_data import load_test_prompts, load_training_data

print("Hugging Face login:", "yes" if get_token() else "no, so HackAPrompt is skipped")

## 2. Data
Test prompts: the test splits of deepset (prompt injections) and jackhhao (jailbreaks), 200 HackAPrompt attacks that worked, and our own prompts. Training data: the safe-guard train split without near copies of test prompts.

In [ ]:
data = load_test_prompts(200)
display(data.groupby(["dataset", "label"], sort=False).size().unstack(fill_value=0)
        .rename(columns={0: "harmless", 1: "attacks"}))

safe_guard = load_training_data(data["text"])

## 3. Models
Six checks, the first only for comparison:
- **Rule-based check:** blocks as soon as one rule fires.
- **Char TF-IDF + SVM:** our tier 1 pick, the same as in the classifier notebook. This is the baseline.
- **+ rules:** plus the nine rules as on/off features.
- **+ text shape:** plus the eight shape features: length, number of words, word length, and the share of punctuation, uppercase letters, digits, special characters (emoji, symbols) and repeated words.
- **+ rules + text shape:** plus both.
- **Markers only + LR:** only the 17 markers, with logistic regression, so that we can read the weight of each marker.

`all` is deepset and jackhhao together.

In [ ]:
bench = Benchmark(data)
bench.run("Rule-based check", rule_check)

models = marker_models()
for name, model in models.items():
    check, seconds = train_sklearn(model, safe_guard)
    bench.run(name, check, training_time=seconds)

## 4. Results
Attacks caught and false alarms per dataset. F1, AUC and time are on deepset and jackhhao together.

In [ ]:
summary(bench.table())

### Weight of each marker
From the markers-only model. A positive weight points to an attack. The rules are 0 or 1; the weight of a shape feature is per standard deviation.

In [ ]:
marker_weights(models["Markers only (rules + text shape) + LR"])

### What the markers changed
Examples for the report: prompts where adding the markers changed the answer of Char TF-IDF + SVM.

In [ ]:
changed(bench.predictions, "Char TF-IDF + SVM", "Char TF-IDF + SVM + rules + text shape")

### Rows for the Overleaf tables

In [ ]:
bench.overleaf_rows()

## 5. Save
Writes `results/complex-feature/classifier_results.csv` and `results/complex-feature/classifier_predictions.csv`.

In [ ]:
bench.save("results/complex-feature")